installed the needed packages:  
`pdfminder` to read pdf file.  
`sklearn_crfsuite` packages contain CRF algorithm

In [1]:
!pip install pdfminer.six sklearn_crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.9 MB/s eta 0:00:00


import needed functions

In [2]:
from pdfminer.high_level import extract_text;
from sklearn_crfsuite import CRF;
from google.colab import drive;
import os;
import re;
import json;

declaration of functions and variables.

To understand this cell, you have to read next cells, it call the function here, it gives contexts.

In [7]:
drive.mount('/content/drive');
pathToInvoiceFolder = "/content/drive/MyDrive/Colab Notebooks/ReadInvoiceAutomation/Invoices";
pathToJsonFolder = "/content/drive/MyDrive/Colab Notebooks/ReadInvoiceAutomation/Json";

def getAllFilesName(path):
  pdfFiles = [];
  allFiles = os.listdir(path);
  for file in allFiles:
    if file.endswith(".pdf"):
      pdfFiles.append(file);
  return pdfFiles;

def readPdfFile(path):
  text = extract_text(path);
  return text;

def tokenizeText(text):
  tokens = re.findall(r"[\w']+|[.,!?;]", text);
  return tokens;

def formatForLabelStudioImport(fileName, text, tokens):
  result = [];
  pos = 0;
  for i, token in enumerate(tokens):
    result.append({
      "from_name": "label",
      "to_name": "text",
      "type": "labels",
      "value": {
        "start": pos,
        "end": pos + len(token),
        "text": token,
        "labels": ["O"]
      }
    });
    pos += len(token) + 1;
  return {
    "data": {
      "text": " ".join(tokens)
    },
    "annotations": [{
      "result": result,
    }],
    "meta": {
      "file_name": fileName
    }
  };

def splitLabel(labeledData: list):
  commonSuffix = ['ltd', 'lld', 'corp']
  X = [];
  Y = [];
  for i, result in enumerate(labeledData):
    X.append({
      "text": result["text"],
      "isText": result['text'].isalpha(),
      'isLower': result['text'].lower(),
      'isTitle': result['text'].istitle(),
      'isUpper': result['text'].isupper(),
      "isNumeric": result['text'].isnumeric(),
      "reletivePos": i/len(labeledData),
      "isCommonSuffix": any(suffix in result['text'].lower() for suffix in commonSuffix),
      'suffix-1': result['text'][-1],
      'suffix-2': result['text'][-2:],
      'suffix-3': result['text'][-3:],
      'prefix-1': result['text'][0],
      'prefix-2': result['text'][:2],
      'prefix-3': result['text'][:3]
    });
    Y.append(result['labels'][0]);
  return X, Y;

def trainModel(X, Y):
  crf = CRF(algorithm='lbfgs', c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True);
  crf.fit(X, Y);
  return crf;

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Part 1
1. get pdf file names
2. read content of the pdf files
3. split the text into tokens
4. prep the file so that we can import this into Universal Data Tool

In [ ]:
# get pdf files' name
filesName = getAllFilesName(pathToInvoiceFolder);

# read content of the pdf files
filesContent = [];
for file in filesName:
  filesContent.append(readPdfFile(pathToInvoiceFolder + "/" + file));

#split text into tokens
filesTokens = [];
for content in filesContent:
  filesTokens.append(tokenizeText(content));

# dump export json
filesJson = [];
for i, file in enumerate(filesTokens):
  filesJson.append(formatForLabelStudioImport(filesName[i], filesContent[i], file));
with open(pathToJsonFolder + "/importData.json", "w") as outfile:
  json.dump(filesJson, outfile);

{'data': {'text': "Date Ready 12 5 2024 Invoice Number Account Code Invoice Date 596243 14252 12 15 2024 Physical location Corporate Couriers Logistics Ltd . Unit 100 8335 Meadow Avenue Burnaby , British Columbia V3N 2W1 Mailing Corporate Couriers Logistics Ltd . P . O . Box 99591 Burnaby RPO Market Crossing Burnaby , BC V5J 0H7 GST HST 826573461 RT0001 TERMS NET 30 DAYS Invoice Submitted To ELEVATE DEVELOPMENT CORP . 3975 North Road Ste 201 Burnaby BC V3J 1S2 Dear Customer , As the holiday season approaches , we want to take a moment to express our gratitude for your continued support and partnership . This is always a busy time for everyone , so we would like to kindly remind you to review any outstanding invoices to ensure timely payments . You can make payments conveniently via the following options 1 . INTERAC E TRANSFER send to accounting corporatecouriers . net 2 . ELECTRONIC FUNDS TRANSFER EFT 3 . CREDIT CARD PAYMENTS VISA MC AMEX If you choose to pay by EFT or with credit card

Part 2: import the exported data into label-studio.
1. to install label-studio:
```
cd <locate-to-your-folder>
python3 -m venv <name-of-the-virtual-environment>
source <name-of-the-virtual-environment>/bin/activate
pip install label-studio
label-studio start
```
2. new we set up label studio with the following lables setting:
```
<View>
  <Labels name="label" toName="text">
    <Label value="O" background="#dddddd"/>
    <Label value="B-COMPANY" background="orange"/>
    <Label value="I-COMPANY" background="gold"/>
    <Label value="B-AMOUNT" background="green"/>
    <Label value="I-AMOUNT" background="lightgreen"/>
  </Labels>
  <Text name="text" value="$text"/>
</View>
```
3. labeling the data then export it. use the option `JSON-MIN`

Part 3: import the labeled data from label studio, then train it
1. import labeled data.
2. split the tokens into X: sample, Y: output
3. train with the given data



In [8]:
# import json data
with open(pathToJsonFolder + "/exportFile.json", "r") as infile:
  labeledData = json.load(infile);

# split the tokens into X and Y
X = [];
Y = [];
for data in labeledData:
  data = data['label'];
  x,y = splitLabel(data);
  X.append(x);
  Y.append(y);

# train with the given data
crf = trainModel(X, Y);

Part 4: use the trained model to actually predict data

In [27]:
idx = 10
prediction = crf.predict_single(X[idx]);
output = [];
for i, y in enumerate(prediction):
  if y != "O":
    output.append(X[idx][i]['text']);
print(output);

[]
